# tp-15 コース適性の類似度（似たコースの実績を借りる特徴量・形状の軸への適性）

**目的**: 馬の過去の実績を、**似たコース**（形状 + 統計プロファイルの類似度）から借りて、コース適性の特徴量を作る。既存の `b_aff_*` は「競馬場×芝ダ」「厳密なコース」などの**完全一致のセルを階層縮約**したもので、似たコースへの転移が無い。初めて走るコースでも、似たコースの実績から適性を推定できるかを測る。

## 特徴量（そのレースより前の実績だけ。同日は含めない）
- `sim_aff`: 似たコースの実績の重みづけ平均（カーネル重み。親 = 馬の全体の平均へ縮約）。実績の指標 res = 0.5 − (着順 − 1)/(頭数 − 1)。
- `sim_n`: 実質的なサンプル数（カーネル重みの合計）。
- `shape_pred` / `shape_se`: 形状の軸（直線の長さ・ゴール前の坂・コーナー数・最初のコーナーまで・高低差・周長・距離）への馬の適性。馬ごとのリッジ回帰（累積で計算）の予測と不確かさ。

## 類似度の作り方（try both）
- G1.0 / G1.5 / G2.5: 形状だけ（標準化した距離のカーネル、τ = 1.0 / 1.5 / 2.5）。
- GS: 形状 + 統計プロファイル（2020〜2021 年のレースから測った 200m ラップの形・最終コーナーからゴールまでの順位変動。**2022 年以降の評価に先読みが入らない**ように、プロファイルは 2021 年までのデータだけ）。

## 評価
LightGBM（調整後、v2 の特徴量セット + 追い切り・厩舎コメント・血統×条件）で、基準に新特徴量を足した版を valid 2022〜2024（3 fold × 2 seed）で比べる。採用基準 = p_win の LogLoss の差の 95% 信頼区間の上限 < 0、連対・複勝が悪化しない（0.0002 以内）、**かつ効果が 0.0002 以上**。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, win_models as wm, zoo, zoo_runner
from src.pipeline.models.compute import detect_compute
from src.research.race import course_similarity as cs

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_p"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP15_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
mlflow.set_experiment("keiba-tp-course-similarity")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
master = cs.load_master(REPO_ROOT / "data/meta/course_master/jra_courses.json")
V = cs.course_vectors(master)
print("コース数", len(V), "| 学習表の行", len(df), "| 形状ベクトルの例:"); display(V.head(3).round(2))

## 統計プロファイル（2020〜2021 年のレースだけ）

In [ ]:
import json as _json
rl = []
for y in (2020, 2021):
    t = pd.read_parquet(REPO_ROOT / f"data/page_reference/tables/{y}/race_result_flat.parquet", columns=["race_id", "venue_code", "surface", "distance", "passing_order", "finish_position", "field_size", "track_condition", "lap_times"])
    rl.append(t)
t = pd.concat(rl); t = t[t["surface"].isin(["芝", "ダート"]) & t["track_condition"].eq("良")]
def last_tok(s):
    k = [x for x in str(s).split("-") if x != ""]
    return int(k[-1]) if k and k[-1].isdigit() else np.nan
t["p4"] = t["passing_order"].apply(last_tok)
t["chg"] = (t["p4"] - t["finish_position"]).abs() / t["field_size"]
race = t.drop_duplicates("race_id")[["race_id", "venue_code", "surface", "distance", "lap_times"]].copy()
race["laps"] = race["lap_times"].apply(lambda s: _json.loads(s) if isinstance(s, str) else None)
rows = []
for (vc, sf, dist), g in race.groupby(["venue_code", "surface", "distance"]):
    L = [x for x in g["laps"] if x]
    if len(L) < 15:
        continue
    nl = pd.Series([len(x) for x in L]).mode()[0]
    A = np.array([x for x in L if len(x) == nl], dtype=float)
    prof = (A - A.mean(1, keepdims=True)).mean(0)
    ch = t[(t["venue_code"] == vc) & (t["surface"] == sf) & (t["distance"] == dist)]["chg"].mean()
    rows.append({"venue_code": vc, "surface": sf, "distance": int(dist), "lap_first_dev": prof[0], "lap_last_dev": prof[-1], "lap_last2_dev": prof[-2], "pos_change": ch})
ST = pd.DataFrame(rows).set_index(["venue_code", "surface", "distance"])
ST = ST.reindex(V.index)
band = pd.cut(V["distance_m"], [0, 1300, 1700, 2100, 9999]).astype(str)
for c in ST.columns:
    ST[c] = ST[c].fillna(ST.groupby([V["is_turf"], band])[c].transform("median")).fillna(ST[c].median())
print("統計プロファイルを測れたコース", int(len(rows)), "/", len(V)); display(ST.head(3).round(3)); ST.to_csv(OUT / "course_stats_pre2022.csv")

In [ ]:
VARIANTS = {"G1.0": dict(tau=1.0), "G1.5": dict(tau=1.5), "G2.5": dict(tau=2.5), "GS": dict(tau=1.5, stats=ST, stat_weight=0.5)}
base_cols = ["race_id", "horse_id", "date", "venue_code", "surface", "distance", "finish_position", "field_size"]
dd = df[base_cols].copy()
FEATS = {}
for name, kw in VARIANTS.items():
    t0 = time.time()
    K = cs.similarity_kernel(V, **kw)
    f = cs.horse_similarity_features(dd, V, K)
    FEATS[name] = f
    f.assign(race_id=df["race_id"].to_numpy(), horse_id=df["horse_id"].to_numpy()).to_parquet(OUT / f"course_sim_{name}.parquet", index=False)
    print(f"{name}: {time.time() - t0:.0f}s  欠損率 {f.isna().mean().round(3).to_dict()}", flush=True)

In [ ]:
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
res0, OOF0 = zoo_runner.run_oof("lgb_extras_ped_s2", "lgb_binary", X0, meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"基準（2 seed）: win {res0['logloss_win'].mean():.5f} top2 {res0['logloss_top2'].mean():.5f} top3 {res0['logloss_top3'].mean():.5f}")


def compare(oof, name):
    m = OOF0.merge(oof, on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": name}
    for t_, k in (("win", 1), ("top2", 2), ("top3", 3)):
        yy = (fin <= k).astype(float)
        bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t_}_b"].to_numpy(), yy), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t_}_v"].to_numpy(), yy), n_boot=1000)
        out[f"{t_}_base"], out[f"{t_}_new"], out[f"Δ{t_}"], out[f"{t_}_hi"] = wm.binary_logloss(m[f"p_{t_}_b"].to_numpy(), yy), wm.binary_logloss(m[f"p_{t_}_v"].to_numpy(), yy), bb["diff"], bb["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δwin"] <= -2e-4 and out["Δtop2"] <= 2e-4 and out["Δtop3"] <= 2e-4)
    return out


ROWS = []
for name, f in FEATS.items():
    t0 = time.time()
    X = pd.concat([X0, f.add_prefix(f"cs_{name}_")], axis=1)
    res, oof = zoo_runner.run_oof(f"lgb_sim_{name}", "lgb_binary", X, meta, folds, SEEDS, THREADS, params=dict(LP))
    ROWS.append(compare(oof, name))
    print(f"{name}: win {res['logloss_win'].mean():.5f} Δ {ROWS[-1]['Δwin']:+.5f} (上限 {ROWS[-1]['win_hi']:+.5f}) ({time.time() - t0:.0f}s)", flush=True)
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5)); R.to_csv(OUT / "similarity_vs_base.csv")

In [ ]:
# 特徴量の寄与（最良の変種で LightGBM の gain）と、新特徴量どうしの関係
bn = R.sort_values("Δwin").index[0]
import lightgbm as lgb
y = (meta["finish_position"].to_numpy() == 1).astype(int)
Xb = pd.concat([X0, FEATS[bn].add_prefix(f"cs_{bn}_")], axis=1)
f = folds[2]
cats = [c for c in Xb.columns if str(Xb[c].dtype) == "category"]
prm = dict({"objective": "binary", "verbosity": -1, "num_threads": THREADS, "seed": 0, "bagging_freq": 1}, **LP["lgb"])
bst = lgb.train(prm, lgb.Dataset(Xb.iloc[f.train_index], label=y[f.train_index], categorical_feature=cats), num_boost_round=LP["n_rounds"])
g = pd.Series(bst.feature_importance("gain"), index=Xb.columns); g = g / g.sum()
new = [c for c in Xb.columns if c.startswith("cs_")]
print("最良の変種:", bn, "| 新特徴量の gain 合計", round(float(g[new].sum()) * 100, 2), "%"); display(g[new].sort_values(ascending=False).round(4))
print("既存のコース適性 b_aff_course / b_aff_venue / b_aff_sd との順位相関:")
c0 = df[["b_aff_course", "b_aff_venue"]].copy(); c0["sim_aff"] = FEATS[bn]["sim_aff"].to_numpy()
display(c0.corr(method="spearman").round(3))
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "results": R.round(6).reset_index().to_dict("records"), "best": bn, "gain_share_pct": float(g[new].sum()) * 100}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-15-course-similarity", tags={"notebook": "tp-15", "git_commit": GIT_COMMIT}):
    for n_, r_ in R.iterrows():
        mlflow.log_metrics({f"delta_win__{n_}": float(r_["Δwin"])})
    for f_ in ("similarity_vs_base.csv", "report.json", "course_stats_pre2022.csv"):
        mlflow.log_artifact(str(OUT / f_), "improve_p")